# Notebook 38 — TON_IoT replication of the input-starvation law

Notebooks 34-37 established, on CICIoT2023, that per-class collapse under pruning is caused by
near-layer-collapse of the input layer under uniform per-layer allocation, mediated by live input
channels, with onset around 100-200 surviving input weights, and that starving the MLP's input layer
reproduces the collapse. This notebook asks whether the law replicates on a second dataset.

**Design.** Five independently initialised CNN1D baselines (channels 64/128; `conv.0` again has 192
weights because the input has one channel) and five MLP baselines (hidden 256/128) are trained on
TON_IoT with the repository's frozen split. Then: conv.0 swept over 0 / 50 / 80 / 90 / 95% sparsity with
the rest at 80% (192 / 96 / 38 / 19 / 10 surviving taps); the MLP input layer starved to 80% and to the
absolute counts 192 / 96 / 38 with the rest at 80%. TON_IoT has ten classes (nine attack types and
normal), so class-count criteria are stated as fractions.

**Gate (stated before running).** (A) CNN damage increases monotonically with conv.0 sparsity
(Spearman >= 0.9) and the 38-tap cell shows mean macro-F1 loss > 0.15 with at least 20% of classes
materially affected in >= 3/5 seeds; (B) the MLP stays intact at 80% (loss < 0.05) and collapses at
38 surviving input weights (loss > 0.15). Any outcome is reported. Resumable per (arch, seed, dose).
GPU runtime required.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, copy, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.utils.prune as prune
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from src.config import CFG, PATHS, set_all_seeds
from src.data import load_raw, clean, temporal_within_capture_split
from src import train as TR, models as M, explain as EXP, mitigate
from src.comnet_audit import assign_validation_tiers, calibration_summary, environment_record, write_json
from src.train import load_anchor, train_model, predict, per_class_recall_table, feature_columns

assert torch.cuda.is_available(), 'switch to a GPU runtime first'
DEVICE = TR.DEVICE
DATASET = 'ton_iot'
SEEDS = list(CFG['seeds']); ANCHOR = int(CFG['anchor_seed'])
OUT = PATHS.tables('comnet')
PRACTICAL_LOSS = 0.10
CNN_DOSES = [0.0, 0.5, 0.8, 0.9, 0.95]                           # conv.0 sparsity -> 192/96/38/19/10 taps
MLP_TARGET_WEIGHTS = [192, 96, 38]                                # absolute surviving input weights (plus the 80% fraction)

KW = {'cnn1d': {'channels': (64, 128)}, 'mlp': {'hidden': (256, 128)}}
BASE = {'cnn1d': 'M0_paired', 'mlp': 'M0'}
from scipy.stats import spearmanr
print('cnn conv.0 doses:', CNN_DOSES, '| mlp target surviving input weights:', MLP_TARGET_WEIGHTS, '(MLP doses computed after loading data)')

In [ ]:
df = clean(load_raw(DATASET, subsample=True, seed=ANCHOR), DATASET)
splits = temporal_within_capture_split(df, seed=ANCHOR)
feat_cols = feature_columns(df)
N_FEAT = len(feat_cols); MLP_INPUT_W = N_FEAT * 256; CNN_INPUT_W = 192
MLP_DOSES = [0.8] + [round(1 - k / MLP_INPUT_W, 6) for k in MLP_TARGET_WEIGHTS]
print(f'{len(df):,} rows | {df.label.nunique()} classes | {N_FEAT} features')
print('MLP input weights:', MLP_INPUT_W, '| doses:', MLP_DOSES, '->', [int(round(MLP_INPUT_W*(1-d))) for d in MLP_DOSES])

In [ ]:
# Pruning policies on the CNN. Fine-tune loop identical to src.compression.prune_and_finetune.
def prunable(model):
    return [(mod, 'weight') for mod in model.modules() if isinstance(mod, (nn.Linear, nn.Conv1d))]

def layer_names(model):
    return {mod: n for n, mod in model.named_modules()}

def apply_layer_amounts(model, amounts):
    # amounts: {module_name: sparsity}; every prunable layer must be named (no silent defaults)
    m = copy.deepcopy(model); names = layer_names(m)
    for mod, name in prunable(m):
        a = amounts[names[mod]]
        if a > 0: prune.l1_unstructured(mod, name=name, amount=float(a)); prune.remove(mod, name)
    return m

def layer_sparsity(model):
    names = layer_names(model); out = {}
    for mod, name in prunable(model):
        w = getattr(mod, name); out[names[mod]] = float((w == 0).float().mean())
    z = sum(int((getattr(mod, n) == 0).sum()) for mod, n in prunable(model)); n_ = sum(getattr(mod, n).numel() for mod, n in prunable(model))
    out['prunable_sparsity'] = z / n_; out['remaining_nonzero_prunable'] = n_ - z
    return out

def finetune_masked(model, seed, *, ft_epochs=8, batch_size=4096, lr=5e-4, verbose=False):
    set_all_seeds(seed)
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(df['label'].to_numpy())
    scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
    t = TR.make_tensors(df, splits, feat_cols, le, scaler); Xtr, ytr = t['train']
    model = model.to(DEVICE)
    masks = {(mod, name): (getattr(mod, name) != 0).float() for mod, name in prunable(model)}
    hooks = [getattr(mod, name).register_hook((lambda mk: (lambda g: g * mk))(mk)) for (mod, name), mk in masks.items()]
    w = TR.tempered_class_weights(ytr.numpy(), len(le.classes_))
    crit = nn.CrossEntropyLoss(weight=w); opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for ep in range(ft_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        if verbose: print(f'    ft epoch {ep}')
    for h in hooks: h.remove()
    with torch.no_grad():
        for mod, name in prunable(model): getattr(mod, name).mul_((getattr(mod, name) != 0).float())
    return model.eval(), le, scaler

def save_ckpt(model, le, scaler, path):
    torch.save({'state_dict': model.state_dict(), 'classes': list(le.classes_), 'feat_cols': feat_cols,
                'scaler_mean': scaler.mean_, 'scaler_scale': scaler.scale_}, path)

# verify the module names we address exist on both architectures
def get_baseline(arch, seed):
    p = PATHS.model(DATASET, arch, BASE[arch], seed)
    if os.path.exists(p):
        return load_anchor(DATASET, arch, BASE[arch], seed, arch_kwargs=KW[arch])
    print(f'  training {arch} baseline seed {seed}')
    m, info = train_model(arch, df, DATASET, splits, seed, epochs=40, patience=6, batch_size=4096, lr=1e-3,
                          compression=BASE[arch], arch_kwargs=KW[arch], save=True, verbose=False)
    return m, info['label_encoder'], info['scaler'], feat_cols

for arch in ('cnn1d', 'mlp'):
    m_, _, _, _ = get_baseline(arch, ANCHOR)
    got = [layer_names(m_)[mod] for mod, _ in prunable(m_)]
    print(f'  {arch}: prunable layers {got}')
    exp = ['conv.0', 'conv.3', 'head'] if arch == 'cnn1d' else ['body.0', 'body.3', 'head']
    assert got == exp, f'{arch} layer names {got} != expected {exp}'
print('input-layer weights: cnn', CNN_INPUT_W, '| mlp', MLP_INPUT_W)

In [ ]:
# Part A + Part B with resume
runs = [('cnn1d', d) for d in CNN_DOSES] + [('mlp', d) for d in MLP_DOSES]
def ckpt_name(arch, d):
    return f'conv0dose{int(round(d*100))}_paired' if arch == 'cnn1d' else f'inputdose{int(round(d*10000))}_mlp_paired'
def amounts_for(arch, d):
    return {'conv.0': d, 'conv.3': 0.8, 'head': 0.8} if arch == 'cnn1d' else {'body.0': d, 'body.3': 0.8, 'head': 0.8}

baseline_val, baseline_test, comp_test, macro, ls_rows = {}, {}, {}, [], []
for arch in ('cnn1d', 'mlp'):
    for seed in SEEDS:
        print(f'\n===== {arch} seed {seed} =====')
        m0, le, scaler, _ = get_baseline(arch, seed)
        yv, pv, _ = predict(m0, df, splits, le, scaler, feat_cols, which='val')
        yt, pt, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test')
        baseline_val[(arch, seed)] = per_class_recall_table(yv, pv, le).set_index('label')['recall']
        baseline_test[(arch, seed)] = per_class_recall_table(yt, pt, le).set_index('label')['recall']
        macro.append({'arch': arch, 'seed': seed, 'dose': -1.0, 'cell': 'M0', 'test_macro_f1': f1_score(yt, pt, average='macro')})
        for a_, d in runs:
            if a_ != arch: continue
            cell = ckpt_name(arch, d); p_c = PATHS.model(DATASET, arch, cell, seed)
            if os.path.exists(p_c):
                mp = M.build(arch, len(feat_cols), len(le.classes_), **KW[arch]).to(DEVICE)
                ck = torch.load(p_c, map_location=DEVICE, weights_only=False)
                mp.load_state_dict(ck['state_dict'] if isinstance(ck, dict) and 'state_dict' in ck else ck); mp.eval(); print(f'  loaded {cell}')
            else:
                mp, _, _ = finetune_masked(apply_layer_amounts(m0, amounts_for(arch, d)), seed); save_ckpt(mp, le, scaler, p_c); print(f'  saved {cell}')
            ls = layer_sparsity(mp); ls_rows.append({'arch': arch, 'seed': seed, 'dose': d, 'cell': cell, **ls})
            yt, pc, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test')
            comp_test[(arch, seed, d)] = per_class_recall_table(yt, pc, le).set_index('label')['recall']
            macro.append({'arch': arch, 'seed': seed, 'dose': d, 'cell': cell, 'test_macro_f1': f1_score(yt, pc, average='macro')})
print('\nall doses complete')

In [ ]:
# Aggregate: per-class effects, dose tables, gate
rows = []
for arch in ('cnn1d', 'mlp'):
    tiers = assign_validation_tiers(pd.DataFrame({s: baseline_val[(arch, s)] for s in SEEDS}))
    for (a_, seed, d), rc in comp_test.items():
        if a_ != arch: continue
        r0 = baseline_test[(arch, seed)]
        for cls in r0.index.intersection(rc.index):
            loss = float(r0.loc[cls] - rc.loc[cls]); band = float(tiers.loc[cls, 'validation_2sd_band'])
            rows.append({'arch': arch, 'seed': seed, 'dose': d, 'class': cls, 'M0_test_recall': float(r0.loc[cls]), 'compressed_test_recall': float(rc.loc[cls]),
                         'recall_loss': loss, 'validation_2sd_band': band, 'material_and_beyond_band': bool((loss > band) and (loss >= PRACTICAL_LOSS))})
eff = pd.DataFrame(rows); assert len(eff) > 0, 'no rows: run the dose cell in this session first'
eff.to_csv(OUT / 'ton_starvation_per_class_effects.csv', index=False)
summ = eff.groupby(['arch', 'dose', 'class']).agg(mean_recall_loss=('recall_loss', 'mean'), affected_frequency=('material_and_beyond_band', 'mean')).reset_index()
summ.to_csv(OUT / 'ton_starvation_per_class_summary.csv', index=False)
mdf = pd.DataFrame(macro); mdf.to_csv(OUT / 'ton_starvation_macro_f1_wide.csv', index=False)
pd.DataFrame(ls_rows).to_csv(OUT / 'ton_starvation_layer_sparsity.csv', index=False)

dose_tab = []
for arch, W in (('cnn1d', CNN_INPUT_W), ('mlp', MLP_INPUT_W)):
    m0m = mdf[(mdf.arch == arch) & (mdf.cell == 'M0')].test_macro_f1.mean()
    for d in (CNN_DOSES if arch == 'cnn1d' else MLP_DOSES):
        g = mdf[(mdf.arch == arch) & (mdf.dose == d)].test_macro_f1
        dose_tab.append({'arch': arch, 'input_layer_sparsity': d, 'surviving_input_weights': int(round(W * (1 - d))),
                         'mean_macro_f1': g.mean(), 'sd_macro_f1': g.std(), 'mean_macro_f1_loss': m0m - g.mean(),
                         'classes_affected_ge3of5': int((summ[(summ.arch == arch) & (summ.dose == d)].affected_frequency >= 0.6).sum())})
dose_tab = pd.DataFrame(dose_tab); dose_tab.to_csv(OUT / 'ton_starvation_dose_response.csv', index=False)
print(dose_tab.round(4).to_string(index=False))

cnn = dose_tab[dose_tab.arch == 'cnn1d']; mlp = dose_tab[dose_tab.arch == 'mlp']
rho_cnn = spearmanr(cnn.input_layer_sparsity, cnn.mean_macro_f1_loss).correlation
rho_mlp = spearmanr(mlp.input_layer_sparsity, mlp.mean_macro_f1_loss).correlation
n_classes = int(df.label.nunique()); min_classes = max(1, int(np.ceil(0.2 * n_classes)))
mlp38 = mlp.iloc[(mlp.surviving_input_weights - 38).abs().argsort().iloc[0]]; mlp80 = mlp[mlp.input_layer_sparsity == 0.8].iloc[0]
cnn38 = cnn[cnn.surviving_input_weights == 38].iloc[0]
verdict = pd.DataFrame([
 {'criterion': 'A_cnn_dose_response_spearman_ge_0.9', 'value': round(float(rho_cnn), 4), 'pass': bool(rho_cnn >= 0.9)},
 {'criterion': f'A_cnn_collapse_at_38_taps_loss_gt_0.15_and_ge_{min_classes}_of_{n_classes}_classes', 'value': f'loss {cnn38.mean_macro_f1_loss:.3f}, {int(cnn38.classes_affected_ge3of5)} classes', 'pass': bool(cnn38.mean_macro_f1_loss > 0.15 and cnn38.classes_affected_ge3of5 >= min_classes)},
 {'criterion': 'B_mlp_collapses_at_38_weights_loss_gt_0.15', 'value': f'loss {mlp38.mean_macro_f1_loss:.3f}, {int(mlp38.classes_affected_ge3of5)} classes', 'pass': bool(mlp38.mean_macro_f1_loss > 0.15)},
 {'criterion': 'B_mlp_intact_at_80pct', 'value': f'loss {mlp80.mean_macro_f1_loss:.3f}', 'pass': bool(mlp80.mean_macro_f1_loss < 0.05)},
 {'criterion': 'ref_mlp_dose_response_spearman', 'value': round(float(rho_mlp), 4), 'pass': ''},
])
print(); print(verdict.to_string(index=False))
print('\nStarvation law replicates on TON_IoT:', bool(verdict[verdict['pass'] != '']['pass'].astype(bool).all()))
verdict.to_csv(OUT / 'ton_starvation_gate_verdict.csv', index=False)
write_json(OUT / 'ton_starvation_environment.json', {'dataset': DATASET, 'n_features': N_FEAT, 'n_classes': int(df.label.nunique()), 'cnn_doses': CNN_DOSES, 'mlp_doses': MLP_DOSES, 'seeds': SEEDS, 'environment': environment_record()})

In [ ]:
# --- Commit + push: main only, this notebook's own files only ---
import subprocess, shutil, glob
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert _b == 'main', f'checked-out branch is {_b!r}; run `git checkout main` first'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred):
    shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
_own = 'notebooks/38_ton_iot_starvation_replication.ipynb'
if os.path.exists(_own):
    d = _json.load(open(_own))
    for c in d.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/ton_starvation_*'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 38: TON_IoT replication - five CNN/MLP baselines, conv.0 dose sweep, MLP input starvation at matched absolute counts, gate verdict'], capture_output=True, text=True)
print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed')
print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)